# Lecture 3

## 3.1 Split operator method (pseudospectral method)

### Motivation: Time dependent Schrödinger equation

$$i\hbar \partial_t |\Psi(t)\rangle = \hat{H}(t) |\Psi(t)\rangle \qquad |\Psi(t=0)\rangle = |\Psi_0\rangle$$

time independent $\hat{H} \rightarrow$ reduction to eigenvalue problem (lecture 1)

$$|\Psi(t)\rangle = \sum_{k} \underbrace{e^{-i E_k t / \hbar}}_{\text{eigenvalues}} \overbrace{\langle k | \Psi_0 \rangle}^{\text{projection of ini state onto eigenstates}} \underbrace{|k\rangle}_{\text{eigenstates}}$$

time-dependent $\hat{H} \Rightarrow$ Eigendecomp. won't work any more

$$\text{formal solution: } |\Psi(t)\rangle = \hat{\mathcal{T}} e^{-i \int dt \hat{H}(t) / \hbar} |\Psi_0\rangle$$

$$\text{numerically: solve by numerical integration in time } (\rightarrow \text{lecture 6})$$


### Special case: 
$H(t) = H_1(t) + H_2(t)$ where the eigenstates of $H_1$ and $H_2$ are indep. of $t$ (only the eigenvalues may be time dependent)

Use discrete time steps $\Delta t$:

$$|\Psi(t)\rangle = e^{-i \Delta t H(t_n)} \dots e^{-i \Delta t H(t_1)} |\Psi_0\rangle \qquad (t_1 = 1)$$

and apply **Trotter expansion** to each term:

$$e^{-i \Delta t (H_1(t) + H_2(t))} = e^{-i \Delta t H_1(t)} e^{-i \Delta t H_2(t)} + \mathcal{O}(\Delta t^2)$$


> **Exercise:** Use Taylor expansion of both sides to 2nd order to check the last equation and show that the Trotter error is $\frac{1}{2} \Delta t^2 [H_1, H_2]$ (i.e. prove the Baker-Campbell-Hausdorff formula to second order)

Assume that the eigendecomp of $H_1, H_2$ is known and given by $(D_1, V_1), (D_2, V_2)$
* $D_1$: evals of $H_1$ on diagonal
* $V_1$: evecs of $H_1$ in columns

### Evolving one time step (think in terms of vectors and matrices)

$$\Psi_{t+\Delta t} = e^{-i H \Delta t} \Psi_t \approx e^{-i H_1(t) \Delta t} e^{-i H_2(t) \Delta t} \Psi_t \qquad \text{($\Psi_t$ in some reference basis)}$$

$$= V_1 V_1^\dagger e^{-i H_1 \Delta t} V_1 V_1^\dagger V_2 V_2^\dagger e^{-i H_2 \Delta t} V_2 V_2^\dagger \Psi_t$$

$$= V_1 e^{-i D_1 \Delta t} V_1^\dagger V_2 e^{-i D_2 \Delta t} V_2^\dagger \Psi_t$$

---

* **For the diagonal matrix operator terms:**
  $$e^{-i D_1 \Delta t} = \begin{pmatrix} e^{-i E_1^{(1)} \Delta t} & & \\ & e^{-i E_2^{(1)} \Delta t} & \\ & & \ddots \end{pmatrix}$$
  $\rightarrow$ component-wise multiplication

* **For the basis transition term:**
  $V_1^\dagger V_2 \rightarrow$ transform from eigenbasis of $H_2$ to eigenbasis of $H_1$

### Algorithm:

* transform $\Psi_0$ into eigenbasis of $H_2$: $\Psi_0 \leftarrow V_2^\dagger \Psi_0$

**repeat $n$ times:**
* $\left[ \begin{aligned} &\text{– apply } e^{-i D_2 \Delta t} : \Psi_1 \leftarrow e^{-i D_2 \Delta t} \Psi_0 \\ &\text{– transform to $H_1$ eigenbasis}: \Psi_1 \leftarrow V_1^\dagger V_2 \Psi_1 \\ &\text{– apply } e^{-i D_1 \Delta t} : \Psi_1 \leftarrow e^{-i D_1 \Delta t} \Psi_1 \\ &\text{– transform back to $H_2$ eigenbasis}: \Psi_1 \leftarrow V_2^\dagger V_1 \Psi_1 \end{aligned} \right.$

* transform back to reference basis: $\Psi_n \leftarrow V_2 \Psi_n$

### Improved scheme ("symmetrized time step"):

$$e^{-i(H_1 + H_2)\Delta t} = e^{-i H_1 \Delta t / 2} e^{-i H_2 \Delta t} e^{-i H_1 \Delta t / 2} + \mathcal{O}(\Delta t^3)$$

$$\text{(prove as exercise)}$$

Has same computational complexity as half step only has to be done once at the beginning and end.

### More advantages / use cases:

* $H_1$ and $H_2$ have analytically known eigendecomp., but $H_1 + H_2$ does not (see below)
* Also works for nonlinear PDEs, e.g.,
  
  $$i \partial_t \psi = -\frac{1}{2} \partial_x^2 \psi + g |\psi|^2 \psi$$

  * $\rightarrow$ applications in optics, cold quantum gases (mean-field equation, "nonlinear Schrödinger equation", Gross-Pitaevskii equation...)

## 3.2 Example: Particle in 1D (Split-step Fourier method)

$$H = -\frac{1}{2} \partial_x^2 + V(x,t) = H_1 + H_2$$

* Breakdown of terms:
    * $-\frac{1}{2} \partial_x^2 \longrightarrow$ time indep., diagonal in momentum space
    * $V(x,t) \longrightarrow$ diagonal in position space

---

eigenfunctions of $H_1$ are plane waves:
$$-\frac{1}{2} \partial_x^2 e^{ipx} = \frac{p^2}{2} e^{ipx}$$

Basis change from position space representation to plane wave basis ($\approx$ k-space) is the Fourier trafo:
$$\phi(p,t) = \frac{1}{\sqrt{2\pi}} \int dx \, e^{-ipx} \psi(x,t)$$

and inverse transformation:
$$\psi(x,t) = \frac{1}{\sqrt{2\pi}} \int dp \, e^{ipx} \phi(p,t)$$

---

### Lattice case: discrete Fourier trafo

* $N$ grid points, spacing $\Delta x = \frac{L}{N}$
* $x_i = -\frac{L}{2}, -\frac{L}{2} + \Delta x, \dots, \frac{L}{2} - \Delta x$

```text
       -L/2      0      L/2
        |---|---|---|---|  (x_i grid spacing)



### Momentum space grid:

$$p = -\frac{N}{2}\Delta p, -\left(\frac{N}{2}-1\right)\Delta p, \dots, -\Delta p, 0, \Delta p, \dots, \left(\frac{N}{2}-1\right)\Delta p$$
$$\text{(assuming even } N\text{)}$$

In Python the momentum grid is shifted:
$$p = 0, \Delta p, \dots, \left(\frac{N}{2}-1\right)\Delta p, -\frac{N}{2}\Delta p, \dots, -\Delta p$$

for reasons that become obvious when examining the Fast Fourier transform.

---

### Discrete FT: Integrals become sums over grid points:

$$px \rightarrow p_k x_j = \frac{2\pi}{L}k \, \Delta x j = \frac{2\pi}{N} k j$$

$$\phi_k(t) = \frac{1}{\sqrt{N}} \sum_{j} \psi_j(t) e^{-\frac{2\pi i}{N} k j}$$

$$\psi_j(t) = \frac{1}{\sqrt{N}} \sum_{k} \phi_k(t) e^{\frac{2\pi i}{N} k j}$$

> **Note on normalization:** The $\frac{1}{\sqrt{N}}$ prefactor is chosen to ensure normalization in vector norm.

<u>note</u>: In scipy implementation FT is without $\frac{1}{\sqrt{N}}$ and inverse FT with $\frac{1}{N}$ prefactor.

### Algorithm:

* $\psi_0(x_j)$ given in position space $\psi_{i,0} = \psi_0(x_i)$
* half time step with $H_2$: $e^{-i V(x_i, t) \Delta t / 2} \psi_i^{(0)}$

$\text{apply } n-1 \text{ times } \begin{cases} 
\text{- Fourier trafo } \phi_k^{(1)} = \text{FT}(\psi^{(0)}) \\
\text{- full time step with } H_1: e^{-i \frac{p_k^2}{2} \Delta t} \phi_k^{(1)} \\
\text{- inverse FT } \psi^{(1)} = \text{FT}^{-1}(\phi^{(1)}) \\
\text{- full time with } H_2
\end{cases}$

* $\text{FT}$
* full time step with $H_1$
* $\text{FT}^{-1}$
* half time step with $H_2$

FT as written in the definition above requires $N$ times calculating a sum over $N$ terms $\rightarrow \sim N^2$ operations per time step.

This can be brought down to $\sim N \log N$ by using the fast Fourier transform (FFT).

### Idea (Cooley Tukey algorithm): 
*(after Cooley & Tukey 1965 but already known to Gauss 1805)*

$$\phi_k = \sum_{j=0}^{N-1} \psi_j e^{-\frac{2\pi i}{N} k j} \quad \left(\text{assuming } N = 2^q\right)$$

$$= \underbrace{\sum_{l=0}^{\frac{N}{2}-1} \psi_{2l} e^{-\frac{2\pi i}{N} k (2l)}}_{\text{even } j \text{ terms}} + \underbrace{\sum_{l=0}^{\frac{N}{2}-1} \psi_{2l+1} e^{-\frac{2\pi i}{N} k (2l+1)}}_{\text{odd } j \text{ terms}}$$

$$= \underbrace{\sum_{l} \psi_{2l} e^{-\frac{2\pi i}{N/2} l k}}_{=: E_k \text{ (FT of vector of even compl. of } \psi\text{)}} + e^{-\frac{2\pi i}{N} k} \underbrace{\sum_{l} \psi_{2l+1} e^{-\frac{2\pi i}{N/2} l k}}_{=: O_k \text{ (FT of vector of odd compl. of } \psi\text{)}}$$

* $\rightarrow$ 2 FTs of vectors of length $N/2 \rightarrow$ iterate
* $\Rightarrow$ terminates after $q = \log_2 N$ steps (only 1 term in sum left)

---

### Now build $\vec{\phi}$ recursively from the end:

2 arrows per comp. in each layer, $\log N$ layers
$$\rightarrow \sim N \log N \text{ operations}$$

Also works for $N \neq 2^q$ but gets slower.

### Remarks:

* **Periodic boundary conditions** are implicit when using FT.
* **Dispersion relation** is different from the finite-difference case:
    * **Split-step:** $H_{\text{kin}} = \frac{p^2}{2} \longrightarrow$ free particle dispersion
    * **Finite difference:** $$H_{\text{kin}} = \frac{1}{2\Delta x^2} \begin{pmatrix} 2 & -1 & & -1 \\ -1 & 2 & \ddots & \\ & \ddots & \ddots & \\ -1 & & 0 & \end{pmatrix}$$

      $$\text{Eigenvalues} \sim \sin\left(\frac{2\pi k}{N}\right)$$

---

### Dispersion Comparison Sketch

```text
    E_kin
      ^         .  <- free particle (parabola)
      |        .
      |   _--_ .    <- lattice disp. rel. (sine wave)
      |  /    \.
______|_/______\______> k
      0       π/2